# 01 · Crear CSV y JSON y subirlos a HDFS

Objetivo: distinguir el archivo de Jupyter (guardado en tu Mac) de su copia en HDFS. Las celdas reescriben únicamente los archivos de ejemplo de tu identificador.

In [ ]:
from pathlib import Path
import json
from clientes import ALUMNO, BASE, hdfs
local = Path("datos") / ALUMNO
local.mkdir(parents=True, exist_ok=True)
ventas = [{"id": 1, "ciudad": "Santander", "importe": 100},
          {"id": 2, "ciudad": "Torrelavega", "importe": 200},
          {"id": 3, "ciudad": "Santander", "importe": 50}]
(local / "ventas.csv").write_text("".join(f"{v['id']},{v['ciudad']},{v['importe']}\n" for v in ventas), encoding="utf-8")
(local / "ventas.jsonl").write_text("".join(json.dumps(v) + "\n" for v in ventas), encoding="utf-8")
print((local / "ventas.csv").read_text())

## Subir los archivos
`hdfs()` ejecuta el mismo comando que `hdfs dfs` y detiene la celda si falla. Cada formato va en una carpeta diferente para que una tabla CSV no intente leer también el JSON.

In [ ]:
for carpeta, archivo in [("ventas_csv", "ventas.csv"), ("ventas_json", "ventas.jsonl")]:
    hdfs("-mkdir", "-p", f"{BASE}/{carpeta}")
    hdfs("-put", "-f", str(local / archivo), f"{BASE}/{carpeta}/")
hdfs("-ls", "-R", BASE)
contenido = hdfs("-cat", f"{BASE}/ventas_csv/ventas.csv")
assert len(contenido.strip().splitlines()) == 3

## Explorar con comandos directos
También puedes usar una celda de comandos:

In [ ]:
!hdfs dfs -ls /user/hdadmin/bda

**Pregunta:** si editas el CSV local, ¿cambia automáticamente HDFS? No: debes volver a subirlo.

El CSV de ejemplo no tiene cabecera ni comas entrecomilladas. El JSON tiene un objeto por línea (JSON Lines). El siguiente cuaderno registra ambos formatos con lectores diferentes.